# Homework 3

## Q1: Try the following experiment:

Open ChatGPT in a private browser window: https://chatgpt.com
Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"
Then, use Kestra's AI Copilot with the same prompt
After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?
- AI Copilot uses a more powerful model
- AI Copilot has access to current Kestra plugin documentation
- AI Copilot uses more tokens
- AI Copilot has internet access


### ChatGPT Output:
```yaml
id: nyc_taxi_csv_to_bigquery
namespace: zoomcamp

variables:
  project_id: "{{ envs.GCP_PROJECT_ID }}"
  dataset: ny_taxi
  table: yellow_tripdata_2024_01
  bucket: "{{ envs.GCP_BUCKET }}"
  file: yellow_tripdata_2024-01.csv.gz

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/{{ vars.file }}"

  - id: create_dataset
    type: io.kestra.plugin.gcp.bigquery.Query
    projectId: "{{ vars.project_id }}"
    serviceAccount: "{{ secret('GCP_CREDS') }}"
    sql: |
      CREATE SCHEMA IF NOT EXISTS `{{ vars.project_id }}.{{ vars.dataset }}`;

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_csv.uri }}"
    destinationTable: "{{ vars.project_id }}.{{ vars.dataset }}.{{ vars.table }}"
    format: CSV
    csvOptions:
      fieldDelimiter: ","
      skipLeadingRows: 1
      allowQuotedNewLines: true
    autodetect: true
    writeDisposition: WRITE_TRUNCATE
    ```

### Kestra AI Output:
```yaml
id: nyc_taxi_data_load
namespace: dev

tasks:
  - id: download_taxi_data
    type: io.kestra.plugin.core.http.Download
    uri: https://raw.githubusercontent.com/plotly/datasets/master/Taxi%20Trip%20Data.csv
    saveAs: nyc_taxi_data.csv

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    # You need to replace 'your_gcp_project.your_bigquery_dataset.nyc_taxi_data' with your actual BigQuery project, dataset, and table.
    # You may also need to configure GCP authentication (e.g., serviceAccount or impersonatedServiceAccount).
    destinationTable: your_gcp_project.your_bigquery_dataset.nyc_taxi_data
    from: "{{outputs.download_taxi_data.uri}}"
    format: CSV
    autodetect: true
    csvOptions:
      skipLeadingRows: 1 # Assuming the CSV has a header row
```

### Q1 Answer is stated around minute 1:07 in this [YouTube video](https://www.youtube.com/watch?v=xUAkcgNIcRI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=29): AI Copilot has access to current Kestra plugin documentation

Built-in Kestra AI Copilot has access to the context, which includes Kestra documentation, which enables it to more correctly build the schema and prevent any syntax errors.

## Q2: 

Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml` in the Kestra UI. Read the execution logs for each.

The non-RAG response about Kestra 1.1 features is best described as:

- Accurate and specific, matching the actual release notes
- Vague, generic, or fabricated — the model guesses from training data
- Empty — the model refuses to answer without context
- Identical to the RAG version

`1_chat_without_rag.yaml` output:

```
INFO 2026-06-29T18:42:39.714911Z ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features that significantly enhanced its capabilities and user experience. Here are at least 5 major features with brief descriptions:

1.  **YAML Autocompletion and Live Validation:** This was a huge step forward for user experience when defining workflows. Instead of just a basic text editor, Kestra 1.1 provided:
    *   **Autocompletion:** As you type, it suggests available tasks, properties, and values, significantly speeding up workflow creation and reducing syntax errors.
    *   **Live Validation:** It immediately highlights syntax errors, missing required properties, or incorrect data types, giving instant feedback and allowing users to fix issues before even attempting to run the workflow. This drastically improved productivity and reduced debugging time.

2.  **Plugin Configuration through UI (with Templating):** Prior to 1.1, configuring plugins often involved editing Kestra's core configuration files. Kestra 1.1 made this much more accessible and manageable by:
    *   **UI-based Configuration:** Allowing users to define and manage plugin configurations (e.g., database connections, S3 buckets, Kafka clusters) directly within the Kestra UI.
    *   **Templating:** Introducing the ability to use variables and templates within these configurations. This meant you could define reusable connection strings and switch between environments (dev, staging, prod) easily without hardcoding values, promoting reusability and maintainability.

3.  **Authentication via OIDC (OpenID Connect):** For organizations looking for more robust and centralized identity management, OIDC support was a critical addition.
    *   **Enterprise-grade Authentication:** Kestra 1.1 enabled integration with popular OIDC providers like Okta, Auth0, Google Identity Platform, and more.
    *   **Single Sign-On (SSO):** This allowed users to leverage their existing enterprise credentials to log into Kestra, streamlining access, improving security, and simplifying user management for administrators.

4.  **Customizable Themes and Branding:** Kestra 1.1 allowed organizations to tailor the look and feel of the Kestra UI to match their own brand identity.
    *   **Branding Customization:** Users could upload custom logos, change primary and secondary colors, and adjust other UI elements.
    *   **Enhanced User Experience:** This made Kestra feel more integrated into an organization's existing tools and improved the overall professional appearance, especially for teams that expose Kestra to a wider audience.

5.  **Blueprint Import/Export:** This feature significantly enhanced the reusability and shareability of workflows.
    *   **Easy Sharing:** Users could export existing workflows (Blueprints) as YAML files.
    *   **Rapid Deployment:** These exported blueprints could then be easily imported into other Kestra instances or shared with other teams, allowing for quick deployment of proven patterns and best practices without starting from scratch. This fostered a collaborative environment and accelerated workflow development.

These features, among others, marked a significant step forward for Kestra, making it more user-friendly, enterprise-ready, and powerful for managing complex data orchestration workflows.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.
```

`2_chat_with_rag.yaml` output:
```
INFO 2026-06-29T18:45:37.768097Z ✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features. Here are at least 5 with brief descriptions:

1.  **New Filters**: The UI filters across Kestra were completely redesigned based on user feedback to be more intuitive and powerful. Users can now choose explicit filter options, reset filters with a single click, save frequently used filter combinations, and customize table column visibility.
2.  **No-Code Dashboard Editor**: Kestra 1.1 extended the no-code multi-panel editor to custom dashboards, allowing users to build and customize dashboards directly from the UI without writing YAML. This enables the creation of data sources, visualizations, and charts using form-based tabs.
3.  **Multi-Agent AI Systems**: AI agents in Kestra can now use other AI agents as tools, facilitating sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
4.  **Fix with AI**: This feature provides AI-powered suggestions when task runs fail, helping users quickly diagnose and resolve issues by offering intelligent recommendations for fixes.
5.  **Human Task**: For Enterprise Edition users, the new `HumanTask` allows for human-in-the-loop workflows. Executions can be paused at this task and will only resume after designated users or group members manually approve them.
6.  **Dozens of New Plugins**: Kestra 1.1 significantly expanded its integration capabilities with numerous new plugins across various categories, including Data & Database (e.g., Liquibase, dlt), SaaS & API (e.g., Airtable, Stripe), Cloud & Infrastructure (e.g., Dataform, AWS CloudWatch), Communication & Notifications (e.g., Resend, Microsoft365 Outlook), and AI Model Providers (e.g., Oracle Cloud Infrastructure GenAI, Cloudflare Workers AI).

🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!
```

### Q2 Answer: Vague, generic, or fabricated — the model guesses from training data

### Q3: Run `4_simple_agent.yaml` with `summary_length = short` (leave the other inputs as defaults).

Open the execution logs and find the token usage logged by the log_token_usage task.

What is the approximate output token count for multilingual_agent?
- 5-15 tokens
- 60-100 tokens
- 200-400 tokens
- 500+ tokens

### Log Output Values for [4_simple_agent_short.yaml](flows/4_simple_agent_short.yaml):

```
INFO 2026-07-03T18:30:15.086037Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 104
- Total tokens: 386

English Brevity Agent:
- Input tokens: 119
- Output tokens: 51
- Total tokens: 170

💡 Tip: Monitor token usage to understand costs and optimize prompts!

```

![flow 4 short](img/flow_4_short_screenshot.png)

### Q3 Answer: A little over 100 tokens

### Q4: With `summary_length = long`, roughly how many times more output tokens does multilingual_agent use compared to the short summary? 
- About the same (within 20%)
- 2-5x more
- 10-20x more
- 50x more

### Log Output Values for [4_simple_agent_long.yaml](flows/4_simple_agent_long.yaml):
```
INFO 2026-07-03T18:38:28.237324Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 192
- Total tokens: 474

English Brevity Agent:
- Input tokens: 207
- Output tokens: 49
- Total tokens: 256

💡 Tip: Monitor token usage to understand costs and optimize prompts!
```

![flow 4 long](img/flow_4_long_screenshot.png)


### Q4 Answer: about 2 times more

### Q5: Open `4_simple_agent.yaml` in the Kestra flow editor. Find the english_brevity task and change its prompt from asking for exactly 1 sentence to asking for exactly 3 sentences.

Save the flow, then run it with `summary_length = long`.

Compare the `english_brevity` output token count to the original 1-sentence version (also with `summary_length = long`). How do they compare?

- About the same (within 20%)
- 2-4x more
- 5-10x more
- 10x+ more

### Log Output Values for [4_simple_agent_long_3_sentences.yaml](flows/4_simple_agent_long_3_sentences.yaml):
```
INFO 2026-07-03T18:55:24.765782Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 190
- Total tokens: 472

English Brevity Agent:
- Input tokens: 205
- Output tokens: 94
- Total tokens: 299

💡 Tip: Monitor token usage to understand costs and optimize prompts!
```

![flow 4 long 3 sentences](img/flow_4_long_3_sentences_screenshot.png)

### Q5: About 2 times

### Q6: Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- Use traditional task-based workflows for predictability and auditability
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data

### Q6 Answer: Use traditional task-based workflows for predictability and auditability